# Notebook \#3: Modelling

# Eviny Charging Curve Classification LSTM 
#### by Sebastian Einar Salas Røkholt
----

### Index
**01 - Setup** </br>
**02 - Data Exploration, Wrangling and Preprocessing**</br>
**03 - Modelling**</br>
**04 - Model Evaluation and Selection**</br>
**+++**</br>

## 01 - Setup

In [22]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit
from typing import Tuple, Union

RANDOM_SEED = 42
TRAIN_MODEL = False
MODEL_PATH = '../Models/LSTM_model_5.pth'

# Notebook settings
%matplotlib inline
pd.options.mode.copy_on_write = True

In [ ]:
# Loads the cleaned dataset with engineered features (minutes_elapsed, temp)
df = pd.read_parquet("../Data/etron55-charging-sessions.parquet")
# These are the features we will be using throughout the notebook
all_features = ["charging_id", "minutes_elapsed", "power", "soc", "temp", "nominal_power"]
fixed_features = ["temp", "nominal_power"]
target_features = ["power", "soc"]
input_features = fixed_features + target_features

# Removes redundant features such as 'energy', 'charging_duration', 
# 'lat' and 'lon' which we don't need for modelling or plotting results
df = df[all_features]
df.head(10)

## 02 - Preparing the dataset for modelling

#### Splitting the data 
The code below splits the dataset into training, validation and test sets. </br>
`GroupShuffleSplit` ensures that a charging session isn't split across multiple sets.

In [3]:
def split_data(df: pd.DataFrame, test_size: float=0.2, validation_size: float=0.1) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Split the dataset into train, validation, and test sets by charging_id grouping.
    
    :param df: Original dataframe containing all charging sessions
    :param test_size: Proportion of data to use for testing
    :param validation_size: Proportion of the remaining data (after test split) to use for validation
    :return: train_df, val_df, test_df DataFrames
    """
    # Defines the test split
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=RANDOM_SEED)
    train_val_idx, test_idx = next(gss_test.split(df, groups=df['charging_id']))
    train_val_df = df.iloc[train_val_idx]

    # Defines the validation split
    validation_size = validation_size / (1 - test_size)  # Adjust validation size based on the remaining dataset after test split
    gss_val = GroupShuffleSplit(n_splits=1, test_size=validation_size, random_state=RANDOM_SEED)
    train_idx, val_idx = next(gss_val.split(train_val_df, groups=train_val_df['charging_id']))

    # Performs the splits on the original dataset
    train_df = train_val_df.iloc[train_idx]
    val_df = train_val_df.iloc[val_idx]
    test_df = df.iloc[test_idx]
    return train_df, val_df, test_df

train_df, val_df, test_df = split_data(df)

#### Data normalisation


In [4]:
# Creates scalers
fixed_features_scaler = MinMaxScaler(feature_range=(0, 1))
power_scaler = MinMaxScaler(feature_range=(0, 1))
soc_scaler = MinMaxScaler(feature_range=(0, 1))

# Fits scalers on training data only
fixed_features_scaler.fit(train_df[fixed_features])
power_scaler.fit(train_df[["power"]])
soc_scaler.fit(train_df[["soc"]])

# Transform train, val, test sets
train_df[fixed_features] = fixed_features_scaler.transform(train_df[fixed_features])
val_df[fixed_features] = fixed_features_scaler.transform(val_df[fixed_features])
test_df[fixed_features] = fixed_features_scaler.transform(test_df[fixed_features])
train_df["power"] = power_scaler.transform(train_df[["power"]])
val_df["power"] = power_scaler.transform(val_df[["power"]])
test_df["power"] = power_scaler.transform(test_df[["power"]])
train_df["soc"] = soc_scaler.transform(train_df[["soc"]])
val_df["soc"] = soc_scaler.transform(val_df[["soc"]])
test_df["soc"] = soc_scaler.transform(test_df[["soc"]])

In [ ]:
def create_sequences(
    df: pd.DataFrame,
    sequence_length: int = 5,
    return_timesteps: bool = False
) -> Union[Tuple[np.ndarray, np.ndarray],
           Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Creates fixed-length input (X) and one-step-ahead target (y) sequences from a dataframe of charging sessions.
    This function groups the dataframe by the charging ID, and for each session, extracts
    `input_features` as model inputs and `target_features` as targets. It slides a 
    window of size `sequence_length` over the session data, producing:
    - `X`: A sequence of shape (sequence_length, num_input_features).
    - `y`: The next step's target values, shape (num_target_features).

    If `return_timesteps=True`, an additional array of time-step indices 
    (minutes_elapsed) is returned for each predicted step (for plotting purposes).
    
    The function relies on the global variables `input_features` and `target_features`
    to determine which columns from the DataFrame go into X and y.
    If a session has fewer than `sequence_length + 1` data points, it will not
    produce any (X, y) pairs.

    Parameters
    ----------
    df : pd.DataFrame
        A DataFrame containing the columns 'charging_id' (identifies the charging session), 'minutes_elapsed' (time index), 
        `input_features` (an array that contains the columns we will be using as input, e.g. ['temp', 'nominal_power', 'power', 'soc']),
        and `target_features` (an array with the columns containing the target variables for the prediction time step, e.g. ['power', 'soc']).
    sequence_length : int, optional
        The number of past observations to include in each input sequence (default 5).
    return_timesteps : bool, optional
        If True, return an additional array of time steps for plotting or analysis (default False).

    Returns
    -------
    (X, y) or (X, y, timesteps) : (np.ndarray, np.ndarray) or (np.ndarray, np.ndarray, np.ndarray) if return_timesteps=True
    """
    all_xs = []
    all_ys = []
    if return_timesteps:
        all_minutes = []  # for plotting

    for session_id, session_df in df.groupby('charging_id', observed=False):
        session_array = session_df[input_features].values
        target_array = session_df[target_features].values
        if return_timesteps:
            session_minutes = session_df["minutes_elapsed"].values  # not scaled, original times

        for i in range(len(session_array) - sequence_length):
            x = session_array[i:i+sequence_length]
            y = target_array[i+sequence_length]
            all_xs.append(x)
            all_ys.append(y)
            if return_timesteps:
                minutes_elapsed_curr_step = session_minutes[i+sequence_length]
                all_minutes.append(minutes_elapsed_curr_step)
    if return_timesteps:
        return np.array(all_xs), np.array(all_ys), np.array(all_minutes)
    else:
        return np.array(all_xs), np.array(all_ys)

# Sets the lookback window for predictions to 15 time steps/minutes
sequence_length = 15
# Create sequences for each set
X_train, y_train = create_sequences(train_df, sequence_length)
X_val, y_val = create_sequences(val_df, sequence_length)
X_test, y_test = create_sequences(test_df, sequence_length)

In [ ]:
print(f"First 5 training sequences: \n(Columns are {input_features})\n{X_train[:5]}")
print(f"\n\nTrue output for first 5 training sequences: \n(Columns are {target_features})\n{y_train[:5]}")

In [7]:
BATCH_SIZE = 128
NUM_WORKERS = 16

# Converts from Numpy arrays to PyTorch tensors
train_features = torch.Tensor(X_train)
train_targets = torch.Tensor(y_train)
val_features = torch.Tensor(X_val)
val_targets = torch.Tensor(y_val)

# Creates TensorDatasets
train_dataset = TensorDataset(train_features, train_targets)
val_dataset = TensorDataset(val_features, val_targets)

# Creates DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

## Modelling

In [8]:
class MultivariateLSTM(nn.Module):
    """
    A 
    Defines the general model architecture and the forward pass for a multivariate 
    LSTM for predicting multiple time-series outputs (e.g., power and SOC).
    
    :param input_size: Number of input features per time step
    :param hidden_layer_size: Dimensionality of the LSTM's hidden state
    :param output_size: Number of output features per time step
    :param num_layers: Number of stacked LSTM layers
    """
    def __init__(self, input_size, hidden_layer_size, output_size, num_layers):
        super(MultivariateLSTM, self).__init__()
        self.hidden_layer_size = hidden_layer_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_layer_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_layer_size, output_size)

    # Forward propagation
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_layer_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_layer_size).to(x.device)
        
        # LSTM layer output
        out, _ = self.lstm(x, (h0, c0))
        
        # Last time step hidden state
        out = self.linear(out[:, -1, :])
        return out


In [9]:
# Defines the training loop (gradient descent + backpropagation)
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = 10,
    initial_lr: float = 0.001,
    plot_loss: bool = True
) -> None:
    """
    Train the given PyTorch model using the provided data loaders.
    
    :param model: PyTorch model to be trained
    :param train_loader: DataLoader for training data
    :param val_loader: DataLoader for validation data
    :param num_epochs: Number of training epochs
    :param initial_lr: Initial learning rate for the Adam optimizer
    :param plot_loss: Whether to plot training vs validation loss
    :return: None
    """    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.train()
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=initial_lr)
    # Define a scheduler that reduces the LR by a factor of 0.1 if the validation loss doesn't improve for 5 epochs
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

    if plot_loss == True:
        # For plotting
        train_losses = list()
        val_losses = list()
    
    # Runs the training loop
    for epoch in range(num_epochs):  # For every pass through the entire dataset
        epoch_train_loss, epoch_val_loss = 0.0, 0.0
        
        for inputs, labels in train_loader:  # For every batch
            inputs, labels = inputs.to(device), labels.to(device)  # Moves data to the training device
            optimizer.zero_grad()  # (Re)sets gradients to 0 so that they don't compound
            outputs = model(inputs)  # Forward pass / Runs predictions
            train_loss = criterion(outputs, labels)  # Calculate the batch loss
            epoch_train_loss += train_loss.item()  # Add loss to this epoch's total training loss
            train_loss.backward()  # Calculates the gradient of the loss function wrt params with backpropagation
            optimizer.step()  # Updates the model parameters based on the gradient, step size and optimization strategy
        
        # For printing the validation loss during training
        model.eval()  # Set model to evaluation mode
        with torch.no_grad():  # No learning here
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss = criterion(outputs, labels)
                epoch_val_loss += val_loss.item()
        
        # Averages the loss per epoch
        avg_epoch_val_loss = epoch_val_loss / len(val_loader)
        avg_epoch_train_loss = epoch_train_loss / len(train_loader)

        # Steps the scheduler
        scheduler.step(avg_epoch_val_loss)

        # Print the epoch's average loss every 5 epochs
        if (epoch + 1) % 5 == 0: 
            curr_lr = optimizer.param_groups[0]["lr"]
            print(f'Epoch {epoch+1}, LR: {curr_lr}, Training loss: {avg_epoch_train_loss},'
                  f'Validation Loss: {avg_epoch_val_loss}')
        if plot_loss == True and epoch > 5:  # I don't want to plot the first few epochs because loss is very high
            # Store loss for plotting purposes
            train_losses.append(avg_epoch_train_loss)
            val_losses.append(avg_epoch_val_loss)

        model.train()  # Set model back to train mode before loading next batch
        
    print('Training complete.')
    if plot_loss == True: 
        # Create a DataFrame from the loss lists
        data = {
            'Epoch': range(1, len(train_losses) + 1),
            'Training Loss': train_losses,
            'Validation Loss': val_losses
        }
        df_losses = pd.DataFrame(data).melt(id_vars=["Epoch"], var_name="Type", value_name="Loss")

        # Plot using Seaborn
        plt.figure(figsize=(10, 6))
        sns.lineplot(data=df_losses, x='Epoch', y='Loss', hue='Type')
        plt.title('Training and Validation Loss Over Number of Training Epochs')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend(title='Loss type')
        plt.show()


In [ ]:
# Setting the model hyperparameters
input_size = len(input_features)  # power, soc, minutes_elapsed, temperature and nominal power
output_size = len(target_features)  # Predicting power and soc for the next time step
HIDDEN_LAYER_SIZE = 60
NUM_HIDDEN_LAYERS = 4
N_TRAINING_EPOCHS = 5

# Defining the network architecture and hyperparameters
model = MultivariateLSTM(input_size, HIDDEN_LAYER_SIZE, output_size, NUM_HIDDEN_LAYERS)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Running the training loop
if TRAIN_MODEL:
    print(f"Training on device {device}...")
    train_model(model, train_loader, val_loader, num_epochs=N_TRAINING_EPOCHS, plot_loss=True)
    torch.save(model.state_dict(), MODEL_PATH)
elif not TRAIN_MODEL and os.path.exists(MODEL_PATH): 
    # Load the model
    model.load_state_dict(torch.load(MODEL_PATH))
    model.eval()
else:
    print(f"TRAIN_MODEL is set to False, but there is no pre-trained model saved at path {MODEL_PATH}")
    print(f"Training a new model on device {device}...")
    train_model(model, train_loader, val_loader, num_epochs=10, plot_loss=True)
    torch.save(model.state_dict(), MODEL_PATH)

## Prediction

#### Prediction on a single test sample

In [ ]:
test_session_ids = test_df["charging_id"].unique()
selected_id = test_session_ids[52]
# Filter the test DataFrame for the selected sessions
selected_sessions_df = test_df[test_df['charging_id'] == selected_id]
print(len(selected_sessions_df))
selected_sessions_df.head()

In [19]:
# Creates sequences with the original timesteps included for plotting purposes
X_test_selected, y_test_selected, minutes_selected = create_sequences(selected_sessions_df, sequence_length, return_timesteps=True)
X_test_selected_torch = torch.tensor(X_test_selected, dtype=torch.float32).to(device)
y_test_selected_torch = torch.tensor(y_test_selected, dtype=torch.float32).to(device)

In [ ]:
# Load the model
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

# Run inference/predictions on test sequence
with torch.no_grad():
    y_pred_selected = model(X_test_selected_torch)

y_pred_selected_np = y_pred_selected.cpu().numpy()
y_true_selected_np = y_test_selected_torch.cpu().numpy()

# Inverse transform
inv_pred_power = power_scaler.inverse_transform(y_pred_selected_np[:, [0]])
inv_pred_soc = soc_scaler.inverse_transform(y_pred_selected_np[:, [1]])
inv_true_power = power_scaler.inverse_transform(y_true_selected_np[:, [0]])
inv_true_soc = soc_scaler.inverse_transform(y_true_selected_np[:, [1]])

full_true_power_scaled = selected_sessions_df["power"].values.reshape(-1, 1)
full_true_soc_scaled   = selected_sessions_df["soc"].values.reshape(-1, 1)

inv_full_true_power = power_scaler.inverse_transform(full_true_power_scaled).ravel()
inv_full_true_soc   = soc_scaler.inverse_transform(full_true_soc_scaled).ravel()

# 2) Prepare arrays for predicted power and SOC
n = len(selected_sessions_df)
full_pred_power = np.empty(n)
full_pred_power[:] = np.nan
full_pred_soc = np.empty(n)
full_pred_soc[:] = np.nan

full_pred_power[sequence_length:] = inv_pred_power.ravel()
full_pred_soc[sequence_length:]   = inv_pred_soc.ravel()

full_minutes = selected_sessions_df["minutes_elapsed"].values

# 3) Plot the full curves
plt.figure(figsize=(12, 6))
plt.plot(full_minutes, inv_full_true_power, label='True Power', color='blue')
plt.plot(full_minutes, full_pred_power, label='Predicted Power', color='red', linestyle='--')
plt.title('Predicted vs. True Power (Entire Session)')
plt.xlabel('Minutes Elapsed')
plt.ylabel('Power')
plt.ylim(0, None)
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(full_minutes, inv_full_true_soc, label='True SOC', color='blue')
plt.plot(full_minutes, full_pred_soc, label='Predicted SOC', color='red', linestyle='--')
plt.title('Predicted vs. True SOC (Entire Session)')
plt.xlabel('Minutes Elapsed')
plt.ylabel('SOC')
plt.legend()
plt.ylim(0, None)
plt.show()

## Anomaly Detection

In [ ]:
def detect_anomalies(model, new_data, threshold):
    model.eval()  # Ensure the model is in evaluation mode
    anomalies = []
    
    # Assuming new_data is preprocessed and in the correct format
    with torch.no_grad():
        for sequence in new_data:
            # Make prediction
            sequence = torch.tensor(sequence, dtype=torch.float).unsqueeze(0).to(device)  # Add batch dimension
            prediction = model(sequence)
            
            # Calculate error
            actual = sequence[:, -1, :2].squeeze().cpu().numpy()  # Last timestep, actual power and soc
            prediction = prediction.squeeze().cpu().numpy()
            error = np.mean((actual - prediction)**2)
            
            # Check if error exceeds threshold
            if error > threshold:
                anomalies.append((sequence.cpu().numpy(), prediction, error))
    
    return anomalies

# Example usage
threshold = 0.01  # This is an example threshold, adjust based on your error analysis
new_data = ...  # Your new data sequences, preprocessed similarly to the training data

anomalies = detect_anomalies(model, new_data, threshold)

print(f"Detected {len(anomalies)} anomalies.")
